# Analyze Your Own Climate Data

Now it's your turn! In this capstone notebook, you'll:

1. **Choose** a weather station anywhere in the US
2. **Fetch** its data from NOAA
3. **Visualize** its climate patterns
4. **Cluster** it alongside 20 reference stations using the same autoencoder from Notebook 3
5. **Discover** where your station falls in "climate space"

This parallels the Birdsong capstone where students record their own birds and cluster them with the curated dataset.

## Step 1: Import Libraries

In [ ]:
import requests
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

## Step 2: Choose Your Station

### How to find a station ID:
1. Visit [NOAA Climate Data Online](https://www.ncdc.noaa.gov/cdo-web/datatools/findstation)
2. Search by city name or zip code
3. Filter by **Daily Summaries** dataset
4. Look for stations with good coverage (many years of data)
5. Copy the station ID (e.g., `USW00012839`)

### Or pick from this list:

| City | Station ID |
|------|-----------|
| Tampa, FL | USW00012842 |
| Charlotte, NC | USW00013881 |
| Detroit, MI | USW00094847 |
| Kansas City, MO | USW00013988 |
| Tucson, AZ | USW00023160 |
| Raleigh, NC | USW00013722 |
| St. Louis, MO | USW00013994 |
| Pittsburgh, PA | USW00094823 |
| Memphis, TN | USW00013893 |
| Albuquerque, NM | USW00023050 |

In [ ]:
# ====== CHANGE THESE ======
MY_STATION_ID = "USW00012842"    # Your chosen station ID
MY_STATION_NAME = "Tampa"         # Give it a name
# ===========================

## Step 3: Fetch Your Data

In [ ]:
def fetch_station(station_id, name):
    """Fetch and clean daily temperature data from NOAA."""
    url = "https://www.ncei.noaa.gov/access/services/data/v1"
    params = {
        "dataset": "daily-summaries",
        "stations": station_id,
        "startDate": "2000-01-01",
        "endDate": "2024-12-31",
        "dataTypes": "TMAX,TMIN,TAVG",
        "units": "metric",
        "format": "json"
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    if not data:
        raise ValueError(f"No data returned for station '{station_id}'. Check that the station ID is valid.")
    df = pd.DataFrame(data)
    df["DATE"] = pd.to_datetime(df["DATE"])
    for col in ["TAVG", "TMAX", "TMIN"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["TAVG"] = df["TAVG"].fillna((df["TMAX"] + df["TMIN"]) / 2)
    df = df.dropna(subset=["TAVG"])
    df["year"] = df["DATE"].dt.year
    df["month"] = df["DATE"].dt.month
    df["dayofyear"] = df["DATE"].dt.dayofyear
    return df

print(f"Fetching data for {MY_STATION_NAME} ({MY_STATION_ID})...")
try:
    my_df = fetch_station(MY_STATION_ID, MY_STATION_NAME)
    print(f"Records: {len(my_df)}")
    print(f"Date range: {my_df['DATE'].min().date()} to {my_df['DATE'].max().date()}")
    print(f"Mean temperature: {my_df['TAVG'].mean():.1f} °C")
    if len(my_df) < 365:
        print(f"WARNING: Only {len(my_df)} records found. This is less than 1 year of data.")
        print("Results may be unreliable. Consider choosing a station with more coverage.")
except requests.exceptions.HTTPError as e:
    print(f"HTTP error: {e}")
    print("Check that your station ID is valid and the NOAA server is available.")
    my_df = pd.DataFrame()
except ValueError as e:
    print(f"Error: {e}")
    print("Double-check your MY_STATION_ID value above.")
    my_df = pd.DataFrame()
except Exception as e:
    print(f"Unexpected error: {e}")
    my_df = pd.DataFrame()
my_df.head()

## Step 4: Explore Your Station

In [ ]:
# Daily temperature — most recent 5 years
recent = my_df[my_df["year"] >= 2020]

plt.figure(figsize=(14, 5))
plt.plot(recent["DATE"], recent["TAVG"], linewidth=0.5, color="steelblue")
plt.fill_between(recent["DATE"], recent["TMIN"], recent["TMAX"], alpha=0.15, color="steelblue")
plt.title(f"{MY_STATION_NAME} Daily Temperature (2020–2024)")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Monthly climatology
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
monthly = my_df.groupby("month")["TAVG"].mean()

plt.figure(figsize=(8, 5))
plt.bar(range(1, 13), monthly.values, color="coral")
plt.xticks(range(1, 13), month_names)
plt.title(f"{MY_STATION_NAME} Monthly Climatology (2000–2024)")
plt.ylabel("Temperature (°C)")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# Temperature heatmap
pivot = my_df.pivot_table(index="year", columns="dayofyear", values="TAVG")

plt.figure(figsize=(15, 7))
sns.heatmap(pivot, cmap="coolwarm", cbar_kws={"label": "°C"}, xticklabels=30, yticklabels=1)
plt.title(f"{MY_STATION_NAME} Daily Temperature Heatmap")
plt.xlabel("Day of Year")
plt.ylabel("Year")
plt.show()

In [ ]:
# Annual average with trend
annual = my_df.groupby("year")["TAVG"].mean()
coeffs = np.polyfit(annual.index, annual.values, 1)
trend = np.poly1d(coeffs)

plt.figure(figsize=(10, 5))
plt.plot(annual.index, annual.values, "o-", color="steelblue", label="Annual Avg")
plt.plot(annual.index, trend(annual.index), "r--", linewidth=2, label="Trend")
plt.title(f"{MY_STATION_NAME} Annual Average Temperature")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Warming rate: {coeffs[0]*10:.2f} °C per decade")

## Step 5: Fetch Reference Stations

We'll fetch the same 20 stations from Notebook 3 to compare against.

In [ ]:
reference_stations = [
    ("Miami",          "USW00012839", "Tropical"),
    ("New York",       "USW00094728", "Continental"),
    ("Chicago",        "USW00094846", "Continental"),
    ("Los Angeles",    "USW00023174", "Mediterranean"),
    ("Seattle",        "USW00024233", "Marine"),
    ("Phoenix",        "USW00023183", "Arid"),
    ("Denver",         "USW00003017", "Semi-Arid"),
    ("Anchorage",      "USW00026451", "Subarctic"),
    ("Honolulu",       "USW00022521", "Tropical"),
    ("Minneapolis",    "USW00014922", "Continental"),
    ("Atlanta",        "USW00013874", "Subtropical"),
    ("Dallas",         "USW00013960", "Subtropical"),
    ("San Francisco",  "USW00023234", "Mediterranean"),
    ("Boston",         "USW00014739", "Continental"),
    ("New Orleans",    "USW00012916", "Subtropical"),
    ("Las Vegas",      "USW00023169", "Arid"),
    ("Portland",       "USW00024229", "Marine"),
    ("Salt Lake City", "USW00024127", "Semi-Arid"),
    ("Jacksonville",   "USW00013889", "Subtropical"),
    ("Nashville",      "USW00013897", "Subtropical"),
]

print("Fetching 20 reference stations...")
ref_data = {}
for i, (name, sid, zone) in enumerate(reference_stations):
    print(f"  [{i+1}/20] {name}...", end=" ")
    try:
        ref_data[name] = fetch_station(sid, name)
        print(f"{len(ref_data[name])} records")
    except Exception as e:
        print(f"FAILED: {e}")
    time.sleep(1)

print(f"\nFetched: {len(ref_data)}/20 stations")

if len(ref_data) < 5:
    print("WARNING: Too few reference stations fetched. Clustering will not be meaningful.")
    print("Check your internet connection or try again later.")

## Step 6: Extract Climate Features

26 features per station: 12 monthly means + 12 monthly stds + annual range + trend slope.

In [ ]:
def extract_features(df):
    """Extract 26 climate features from a station's daily data."""
    monthly = df.groupby("month")["TAVG"]
    # Use reindex to guarantee 12 values (months 1-12), even if some months are missing
    monthly_means = monthly.mean().reindex(range(1, 13)).values
    monthly_stds = monthly.std().reindex(range(1, 13)).values
    annual_range = np.nanmax(monthly_means) - np.nanmin(monthly_means)
    annual = df.groupby("year")["TAVG"].mean().dropna()
    slope = np.polyfit(annual.index, annual.values, 1)[0] if len(annual) > 1 else 0.0
    return np.concatenate([monthly_means, monthly_stds, [annual_range, slope]])

# Extract for all reference stations
all_features = {}
for name in ref_data:
    all_features[name] = extract_features(ref_data[name])

# Check for name collision: if student's station name matches a reference station
my_name = MY_STATION_NAME
if my_name in all_features:
    my_name = MY_STATION_NAME + " (yours)"
    print(f"Note: '{MY_STATION_NAME}' matches a reference station. Using '{my_name}' to distinguish yours.")

# Extract for YOUR station
all_features[my_name] = extract_features(my_df)

feature_matrix = np.array(list(all_features.values()))
station_names = list(all_features.keys())
print(f"Feature matrix: {feature_matrix.shape[0]} stations x {feature_matrix.shape[1]} features")
print(f"Your station ({my_name}) is included!")

## Step 7: Train the Autoencoder

In [ ]:
# Preprocessing
scaler = StandardScaler()
X_scaled = scaler.fit_transform(feature_matrix)
X_tensor = torch.FloatTensor(X_scaled)

# Same autoencoder as Notebook 3
class ClimateAutoencoder(nn.Module):
    def __init__(self, input_dim=26, bottleneck=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, input_dim),
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

model = ClimateAutoencoder()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

print("Training autoencoder...")
for epoch in range(5000):
    output = model(X_tensor)
    loss = criterion(output, X_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 1000 == 0:
        print(f"  Epoch {epoch}: loss = {loss.item():.6f}")

print(f"  Final loss: {loss.item():.6f}")

## Step 8: Where Does Your Station Fall?

In [ ]:
model.eval()
with torch.no_grad():
    embeddings = model.encoder(X_tensor).numpy()

zone_lookup = {name: zone for name, _, zone in reference_stations}
zone_colors = {
    "Tropical": "#e74c3c", "Subtropical": "#e67e22", "Continental": "#3498db",
    "Mediterranean": "#2ecc71", "Marine": "#9b59b6", "Arid": "#f1c40f",
    "Semi-Arid": "#d4ac0d", "Subarctic": "#1abc9c",
}

plt.figure(figsize=(14, 9))

# Plot reference stations
for i, name in enumerate(station_names):
    if name == my_name:
        continue
    zone = zone_lookup.get(name, "Unknown")
    color = zone_colors.get(zone, "gray")
    plt.scatter(embeddings[i, 0], embeddings[i, 1], c=color, s=80,
                zorder=3, edgecolors="black", linewidth=0.5, alpha=0.7)
    plt.annotate(name, (embeddings[i, 0], embeddings[i, 1]),
                 textcoords="offset points", xytext=(6, 6), fontsize=8, alpha=0.7)

# Plot YOUR station as a big star
my_idx = station_names.index(my_name)
plt.scatter(embeddings[my_idx, 0], embeddings[my_idx, 1],
            c="gold", s=400, marker="*", zorder=5, edgecolors="black", linewidth=1.5)
plt.annotate(f"  {my_name} (YOU)", (embeddings[my_idx, 0], embeddings[my_idx, 1]),
             fontsize=12, fontweight="bold", color="darkred")

# Legend
for zone, color in zone_colors.items():
    plt.scatter([], [], c=color, s=80, label=zone, edgecolors="black", linewidth=0.5)
plt.scatter([], [], c="gold", s=200, marker="*", label="Your Station", edgecolors="black")
plt.legend(title="Climate Zone", loc="best")

plt.title("Your Station in Climate Space (Autoencoder 2D Embedding)")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True, alpha=0.3)
plt.show()

## Step 9: Interpretation

Look at where your star landed:
- **Which reference stations is it closest to?**
- **Does the grouping match what you expected?**
- **What climate zone does it seem to belong to?**

The autoencoder placed your station in 2D space based on 26 climate features — the same technique used for birdsong species clustering!

## Bonus: Compare Two Decades

Has your station's climate shifted? Extract features separately for 2000–2012 and 2013–2024.

In [ ]:
# Split into early and recent periods
my_early = my_df[my_df["year"] <= 2012]
my_recent = my_df[my_df["year"] >= 2013]

if len(my_early) > 365 and len(my_recent) > 365:
    feat_early = extract_features(my_early)
    feat_recent = extract_features(my_recent)

    # Add temporal features to the existing feature matrix and re-scale
    all_feat_temporal = feature_matrix.tolist()
    all_feat_temporal.append(feat_early)
    all_feat_temporal.append(feat_recent)
    names_temporal = station_names + [f"{my_name} (2000-12)", f"{my_name} (2013-24)"]

    # Re-scale with a new scaler to include the temporal features
    X_temp = torch.FloatTensor(StandardScaler().fit_transform(np.array(all_feat_temporal)))

    # Reuse the already-trained model's encoder (no need to train a new one)
    model.eval()
    with torch.no_grad():
        emb2 = model.encoder(X_temp).numpy()

    plt.figure(figsize=(14, 9))
    for i, name in enumerate(names_temporal[:-2]):
        zone = zone_lookup.get(name, "Unknown")
        color = zone_colors.get(zone, "gray")
        plt.scatter(emb2[i, 0], emb2[i, 1], c=color, s=60, zorder=3,
                    edgecolors="black", linewidth=0.5, alpha=0.5)
        plt.annotate(name, (emb2[i, 0], emb2[i, 1]),
                     textcoords="offset points", xytext=(5, 5), fontsize=7, alpha=0.5)

    # Early period
    plt.scatter(emb2[-2, 0], emb2[-2, 1], c="blue", s=300, marker="*",
                zorder=5, edgecolors="black", linewidth=1.5)
    plt.annotate(f"  {my_name} (2000-12)", (emb2[-2, 0], emb2[-2, 1]),
                 fontsize=11, fontweight="bold", color="blue")
    # Recent period
    plt.scatter(emb2[-1, 0], emb2[-1, 1], c="red", s=300, marker="*",
                zorder=5, edgecolors="black", linewidth=1.5)
    plt.annotate(f"  {my_name} (2013-24)", (emb2[-1, 0], emb2[-1, 1]),
                 fontsize=11, fontweight="bold", color="red")

    plt.title(f"Climate Shift: {my_name} Early vs. Recent")
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.grid(True, alpha=0.3)
    plt.show()

    print("Blue star = 2000-2012 climate. Red star = 2013-2024 climate.")
    print("If the red star moved, your station's climate has shifted!")
else:
    print("Not enough data for temporal comparison.")

## Bonus: Save to Google Drive (Colab)

Uncomment the lines below to save your results.

In [ ]:
# Uncomment to save to Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# my_df.to_csv(f'/content/drive/MyDrive/{MY_STATION_NAME}_climate_data.csv', index=False)
# print(f"Saved to Google Drive: {MY_STATION_NAME}_climate_data.csv")

## Congratulations!

You've completed the full FAU Geoscience & Climate Data curriculum! Across 6 notebooks, you learned:

1. **Python fundamentals** — variables, lists, loops, functions
2. **Climate data processing** — fetching from APIs, cleaning, time series
3. **Visual pattern recognition** — heatmaps, climatology charts, seasonal cycles
4. **Machine learning** — feature extraction, autoencoders, unsupervised clustering
5. **Spatial data** — satellite imagery, gridded reanalysis, cartopy maps
6. **Independent research** — choosing and analyzing your own data

These are the same tools and techniques used by working climate scientists and data scientists. Keep exploring!